## Modern LLMs

### Plan

1. Modern language models. Article: [Aligning language models to follow instructions](https://openai.com/index/instruction-following/)
1. The difference between pre-trained models and fine-tuned models using the example of Llama 3.2-1B:
- Architecture
- Tokenizers
- Chat template
3. Supervised finetuning
4. RLHF


In [ ]:
# Environment for week08: Modern LLMs: base vs instruct, SFT, RLHF, DPO.
# Colab or a fresh virtualenv: run this cell once. Locally: `uv sync --group week08` in the repo root instead.
%pip install -q "torch>=2.4,<3" "transformers>=4.56,<5" "accelerate>=1.0"

## Modern LLMs

All models now undergo three stages of training: pre-training, supervised fine-tuning, and RLHF. Let's go through each of them:

### Pre-training
Language models learn to perform the task of next token prediction. To do this, huge datasets are usually collected automatically. Standard data sources include website parsing (followed by removal of HTML tags and other elements), book recognition, and so on. Examples of datasets: [c4](https://huggingface.co/datasets/allenai/c4), [RedPajama](https://github.com/togethercomputer/RedPajama-Data), [wikipedia](https://huggingface.co/datasets/wikimedia/wikipedia)

There are many benchmarks for testing their capabilities, such as [hellaswag](https://arxiv.org/abs/1905.07830), [OpenBookQA](https://huggingface.co/datasets/allenai/openbookqa), [WinoGrande](https://huggingface.co/datasets/allenai/winogrande). In such benchmarks, they are most often asked to either generate a multiple-choice answer or generate a continuation (completion).

Such models have excellent knowledge of the world because they have seen a lot of data, but they have a problem: they cannot respond in a format that is useful to us, humans. Let's look at the example of Llama-3.2-1B.

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Попробуем попросить ее решить простую задачку

In [2]:
pipe_pretrained = pipeline(
    "text-generation",
    model="unsloth/Llama-3.2-1B",
    max_new_tokens=512,
    device=device,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda


In [7]:
task = "Find the value of n in the equation n + 2 = 6."

In [8]:
pipe_pretrained(task)

[{'generated_text': "Find the value of n in the equation n + 2 = 6. If n is an integer, what is the value of n?\nWe have to find the value of n in the equation n + 2 = 6. So, we can write the given equation as\nn + 2 = 6\nNow, we need to find the value of n such that the equation is satisfied. To do this, we can use the rules of algebra.\nIf the equation is equal to zero, the value of the variable will be zero.\nIf the equation is greater than zero, the value of the variable will be positive.\nIf the equation is less than zero, the value of the variable will be negative.\nNow, let's check the given equation and see what happens when we substitute the value of n.\nSubstituting the value of n = 4 in the given equation, we get\n4 + 2 = 6\nSo, the value of n is 4.\nSo, the value of n is 4.\nIf n is an integer, what is the value of n?\nn = 4\nIf n is an integer, what is the value of n?\nn = 5\nIf n is an integer, what is the value of n?\nn = 6\nIf n is an integer, what is the value of n?\nn

1. Learn to read unformatted strings.  
1. Something clearly went wrong. Actually, this can be easily solved by composing the prompt in the form of a completion of the text:

In [5]:
task_fixed = "Find the value of n in the equation n + 2 = 6. Answer: n = "

In [9]:
pipe_pretrained(task_fixed)

[{'generated_text': 'Найдите значение n в уравнении n + 2 = 6. Ответ: n = 4\n1. A. n + 2 = 6\n2. B. 2n + 2 = 6\n3. C. 2n = 4\n4. D. 2n + 2 = 6\n5. E. n + 2 = 6\n6. F. 2n = 4\n7. G. 2n + 2 = 6\n8. H. 2n + 2 = 6\n9. I. 2n = 4\n10. J. 2n + 2 = 6\n11. K. 2n = 4\n12. L. 2n + 2 = 6\n13. M. 2n = 4\n14. N. 2n + 2 = 6\n15. O. 2n = 4\n16. P. 2n + 2 = 6\n17. Q. 2n = 4\n18. R. 2n + 2 = 6\n19. S. 2n = 4\n20. T. 2n + 2 = 6\n21. U. 2n = 4\n22. V. 2n + 2 = 6\n23. W. 2n = 4\n24. X. 2n + 2 = 6\n25. Y. 2n = 4\n26. Z. 2n + 2 = 6\n27. AA. 2n = 4\n28. AB. 2n + 2 = 6\n29. AC. 2n + 2 = 6\n30. AD. 2n = 4\n31. AE. 2n + 2 = 6\n32. AF. 2n = 4\n33. AG. 2n + 2 = 6\n34. AH. 2n = 4\n35. AI. 2n + 2 = 6\n36. AJ. 2n = 4\n37. AK. 2n + 2 = 6\n38. AL. 2n = 4\n39. AM. 2n + 2 = 6\n40. AN. 2n = 4\n41. AO. 2n +'}]

That's better, but it's still far from how GigaChat behaves. What's the reason?

**Models designed to interact with users go through several stages of training. These stages are collectively referred to as Alignment (although everyone now attaches different meanings to this concept).**

This name comes from the article [Aligning language models to follow instructions](https://openai.com/index/instruction-following/).

Let's see how a chat model copes with such tasks:

In [10]:
pipe = pipeline(
    "text-generation",
    model="unsloth/Llama-3.2-1B-Instruct",
    max_new_tokens=512,
    device=device,
)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Device set to use cuda


However, we will have to change the input data format slightly, as chat models accept data in dialogue format.

In [11]:
messages = [
    {"role": "user", "content": "Find the value of n in the equation n + 2 = 6."},
]

In [12]:
pipe(messages)

[{'generated_text': [{'role': 'user',
    'content': 'Find the value of n in the equation n + 2 = 6.'},
   {'role': 'assistant',
    'content': 'To find the value of n, we need to isolate n on one side of the equation.\n\nGiven equation: n + 2 = 6\n\nSubtract 2 from both sides of the equation:\nn + 2 - 2 = 6 - 2\nn = 4\n\nSo, the value of n is 4.'}]}]

Here we see the kind of response we have come to expect from GigaChat. The most interesting thing is that there are no differences in the models' architectures at all.

In [13]:
pipe.model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [14]:
pipe_pretrained.model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,)

In [16]:
str(pipe_pretrained.model) == str(pipe.model)

True

Except for the template

In [28]:
pipe.tokenizer.apply_chat_template(messages, tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 13 Nov 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nFind the value of n in the equation n + 2 = 6.<|eot_id|>'

In [19]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer_pretrained = AutoTokenizer.from_pretrained(
    "unsloth/Llama-3.2-1B",
    device=device,
)
model_pretrained = AutoModelForCausalLM.from_pretrained(
    "unsloth/Llama-3.2-1B",
    device_map="auto", # otherwise bs error because of no adaptation still (???) probably because of Grouped-Query Attention (which is also 1 y.o. (?????))
)

### Why there is such a difference in completions then?

---


## Supervised Fine-Tuning

How do you train humans to do something? You give them a ~ChatGPT~ task and show a good solution to that task.

The idea here is the same: let's collect a dataset in the format of a dialogue between a user and an assistant. The user asks a question, and the assistant answers it.

Ideally (as it was done in the original article), the answers should be written by expert humans in the certain domain (e.g. Yandex Prompt Engineers or ~Talibank~ Tinkoff Product Analysts.

This way, the model learns the response format and uses the knowledge it gained in the previous stage to help the user.

Examples of such datasets: [UltraChat] (https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k), [No-Robots](https://huggingface.co/datasets/HuggingFaceH4/no_robots), [MathInstruct](https://huggingface.co/datasets/TIGER-Lab/MathInstruct).

To collect these datasets, large companies (OpenAI, Meta, Yandex, Sber and others) hired entire departments consisting of professional editors to write instructions and answers to them.

Next, the data collected undergoes standard training as in the previous stage (next-token prediction).

![SFT](https://habrastorage.org/webt/79/dr/np/79drnpe2spuzrttprryqkpjt8tk.jpeg)

### Actually, such data can now be collected without expensive markup by editors.

There are several approaches, such as [Alpaca](https://crfm.stanford.edu/2023/03/13/alpaca.html), that allow data to be collected without them.

Alpaca works as follows:

- Select 175 human-written instructions from the self-instruct dataset
- Prompt GPT-3.5 (at that time it was SOTA) to generate more similar instructions
- Learn from the data obtained
- Gain the ChatGPT distillation


![Alpaca](https://crfm.stanford.edu/static/img/posts/2023-03-13-alpaca/alpaca_main.jpg)

But there are other methods, such as [Instruction Pre-Training](https://arxiv.org/abs/2406.14491), [Mammoth](https://arxiv.org/abs/2405.03548)

They boil down to the fact that instructions can be generated from the same data on which the model is pre-trained.

## Preference Learning (RLHF)

But wait, there's more! Although SFT already provides decent quality, the real breakthrough came with the RLHF method, which was first successfully applied by OpenAI in [InstructGPT](https://openai.com/index/instruction-following/).

This method proposes not only increasing the probability of “good” answers, but also decreasing the probability of “bad” ones, such as toxic, dangerous, or deceitful answers.

What is the essence of this approach?

- Several answer options are generated for a single query by a single model, for example, by changing the generation parameters.
- Responses are ranked by us, humans: thus, a response without any “flaws” will be preferred over one with them.
- Next, a reward model is trained on the responses, which gives a larger reward to a “better” response.
- After that, reinforcement learning methods, namely PPO, are used to maximize the reward.

![RLHF] (https://images.ctfassets.net/kftzwdyauwt9/12CHOYcRkqSuwzxRp46fZD/928a06fd1dae351a8edcf6c82fbda72e/Methods_Diagram_light_mode.jpg?w=3840&q=80&fm=webp)

People preferred the responses of the smaller model trained with RLHF to those of the larger model.

![img](https://dz2cdn1.dzone.com/storage/temp/16620244-1672726052844.png)

# DPO


However, PPO is a very expensive method with many hyperparameters. It is very easy to make mistakes in its implementation, which are hell to debug.

In 2023, an article titled [Direct Preference Optimization](https://arxiv.org/abs/2305.18290) was published.

Through complex maths, the authors concluded that the optimal policy is expressed through the function itself (which is why the article is called Your Language Model is Secretly a Reward Model), so it is possible to avoid problems with PPO and RL in general.

![DPO](https://ghost.oxen.ai/content/images/2024/01/Screenshot-2024-01-26-at-3.49.01-PM.png)

All these (and other) methods are available in the [TRL](https://huggingface.co/docs/trl/en/index) library.

- [SFT Trainer](https://huggingface.co/docs/trl/en/sft_trainer)

- [Reward Modelling](https://huggingface.co/docs/trl/en/reward_trainer)

- [PPO Trainer](https://huggingface.co/docs/trl/en/ppo_trainer)

- [DPO Trainer](https://huggingface.co/docs/trl/en/dpo_trainer)

![TRL](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)

## How to evaluate the quality of a model and determine which one is right for you?

## There are many benchmarks that test specific model skills.

Examples of benchmarks:

- [MT-Bench](https://arxiv.org/abs/2306.05685): completions are generated for multi-step dialogues (several human-assistant transitions) and then evaluated using GPT4.
- [AlpacaEval](https://tatsu-lab.github.io/alpaca_eval/): the model's response is compared with the GPT4 response, then evaluated using another model. The win rate is then calculated.


- [ChatBot Arena](https://lmarena.ai): Users are given the opportunity to chat with different models (including paid ones) and ask them questions. Then, the user is asked to choose the answer they like best out of two. Based on this, a model rating is constructed.

- [LLM Arena](https://llmarena.ru): The same, but in Russian.

Current Arena Leaderboard

[👉🏻👉🏻👉🏻 Arena 👈🏻👈🏻👈🏻](https://lmarena.ai/leaderboard/text)

## Which model should you choose?

### Almost always, preference should be given to models that are trained to respond in a chat format. That is, those that have passed the alignment stage.

Training your model from scratch is almost always financially unprofitable, as well as impractical, unless, of course, you work for a huge company that has the resources to do so.


Modern models are relatively “small” in size ([gemma2 9b] (https://huggingface.co/google/gemma-2-9b-it), [llama-3.1 8B](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct)) perform well on general tasks, significantly better than “old” models, even large ones.

Large models ([gemma2 27b](https://huggingface.co/google/gemma-2-27b-it), [llama3.1 405b] (https://huggingface.co/meta-llama/Llama-3.1-405B-Instruct)) often do not even need to be additionally trained for specific tasks (such as summarization or classification); it is enough to write the model task correctly and describe the expected output format.

You can read more about prompt engineering [here](https://platform.openai.com/docs/guides/prompt-engineering).